<a href="https://colab.research.google.com/github/mlopsvis/MiniTransformer/blob/FT_Main_MiniTransformer_Upload/MiniTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from collections import defaultdict

# Set random seed for reproducibility
random.seed(42)
torch.manual_seed(42)

# Define a small vocabulary
vocab = ["cat", "sat", "on", "the", "mat", "dog", "ran", "to", "yard", "bird", "flew", "in", "sky", "fish", "swam", "under", "water"]
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}
vocab_size = len(vocab)
embedding_dim = 16

# Generate synthetic sentence pairs
def generate_sentences(num_sentences=200):
    templates = [
        ["cat", "sat", "on", "the", "mat"],
        ["dog", "ran", "to", "the", "yard"],
        ["bird", "flew", "in", "the", "sky"],
        ["fish", "swam", "under", "the", "water"]
    ]
    sentences = []
    for _ in range(num_sentences):
        template = random.choice(templates)
        sentences.append(template)
    return sentences



In [3]:

# Prepare training data
def prepare_data(sentences):
    inputs = []
    targets = []
    for sentence in sentences:
        input_seq = sentence[:4]
        target_word = sentence[4]
        input_ids = [word2idx[word] for word in input_seq]
        target_id = word2idx[target_word]
        inputs.append(input_ids)
        targets.append(target_id)
    return torch.tensor(inputs), torch.tensor(targets)


In [4]:

# Define a simple transformer-like model
class MiniTransformer(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(MiniTransformer, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.attn = nn.MultiheadAttention(embed_dim=embedding_dim, num_heads=1, batch_first=True)
        self.fc = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        attn_output, _ = self.attn(x, x, x)
        x = attn_output[:, -1, :]  # Use the last token's output
        logits = self.fc(x)
        return logits


In [5]:
# Generate and prepare data
sentences = generate_sentences(300)
inputs, targets = prepare_data(sentences)

# Initialize model, loss, and optimizer
model = MiniTransformer(vocab_size, embedding_dim)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Train the model
model.train()
for epoch in range(20):
    optimizer.zero_grad()
    logits = model(inputs)
    loss = criterion(logits, targets)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# Evaluate on a test input
model.eval()
test_input = torch.tensor([[word2idx[word] for word in ["cat", "swam", "under", "the"]]])
with torch.no_grad():
    logits = model(test_input)
    probs = F.softmax(logits, dim=-1)
    predicted_idx = torch.argmax(probs, dim=-1).item()
    print(f"Predicted next word: {idx2word[predicted_idx]}")



Epoch 5, Loss: 2.0205
Epoch 10, Loss: 0.8881
Epoch 15, Loss: 0.1219
Epoch 20, Loss: 0.0118
Predicted next word: mat


In [8]:
test_input = torch.tensor([[word2idx[word] for word in ["dog", "swam", "under", "the"]]])
with torch.no_grad():
    logits = model(test_input)
    probs = F.softmax(logits, dim=-1)
    predicted_idx = torch.argmax(probs, dim=-1).item()
    print(f"Predicted next word: {idx2word[predicted_idx]}")


Predicted next word: yard
